# C6 — per-batch action-space ceiling gate

**The disciplined pre-condition before training any learner in the per-batch space.**

The deployable-residual result (`Q_R1_DEPLOYABLE_RESIDUAL_ADJUDICATION_2026-07-22`) closed the
coarse **4⁸ = 65,536** weekly-count action space: the frozen c256 structured comparator hits the
**exact global optimum in 23/24 campaigns** at κ=0.90, so there is no convertible headroom there and
no learner can win. That result explicitly does **not** cover the richer **per-batch** space.

The per-batch space is **8⁸ = 16,777,216**: each week is one of 8 *ordered* three-batch patterns
instead of one of 4 *counts*. We verified locally that intra-week ordering **does** move the cohort
ReT (102,765 / 196,608 random calendars differ from their count-projection, worst Δ = 0.183), so this
space is genuinely richer — the door is **not** closed by construction.

This notebook measures the **clairvoyant ceiling** of that richer space: for each burned campaign, the
exact maximum cohort ReT over all 16.7M per-batch calendars, minus what the frozen c256 comparator
achieved. Because the ceiling is the best any policy could do with hindsight, it **upper-bounds any
learner**:

* if `LCB95(ceiling − frozen) < SESOI` in every stratum → **the last action-space door is closed**,
  no PPO/GRU can win here, and we stop with the strongest possible negative — *without training
  anything*;
* if some stratum clears the gate → headroom exists, and **only then** is a per-batch PPO run a
  justified experiment (Phase 2, a separate notebook).

Nothing here trains a learner. It is a gate, not a result. Burned roots, `BURNED_DEVELOPMENT_NO_CLAIM`.

In [ ]:
# ============================================================
# 0) Configuration  --  edit here; then Run All
# ============================================================
# RUN_PROFILE:
#   "debug"    3 campaigns, 500k sampled calendars/campaign  -> ~1 min smoke test
#   "serious"  all 48 campaigns, EXACT 8^8 enumeration        -> the real gate
RUN_PROFILE = "serious"

GIT_URL    = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "codex/q-r1-comparator-reconciliation"   # has the frozen c256 Pareto
REPO_DIR_COLAB  = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

RUNTIME_PIP_PACKAGES = ["simpy>=4.1", "numpy>=1.26"]

# Frozen comparator artifacts already committed on the branch above.
PARETO_REL = "results/q_r1/comparator_v2_frozen_pareto_c256_v1/pareto_merged/result.json"

# Gate.
SESOI = 0.01
DEPLOYABLE_GATE = 0.015          # the route's GO threshold for opening a learner
BOOTSTRAP_DRAWS = 10_000
BOOTSTRAP_SEED  = 20_260_722

# Compute.
CHUNK   = 65_536                 # calendars per DES call (peak RSS ~1.1 GB per worker)
N_WORKERS = 4                    # Kaggle CPU kernels expose 4 cores
DEBUG_SAMPLE = 500_000
DEBUG_CAMPAIGNS = 3

# Campaign reconstruction (must match the Pareto exactly).
CAMPAIGNS_PER_HISTORY = 12
REGIME_PERSISTENCE = 0.90
DOMINANT_SHARE = 0.90
MODE_TO_KAPPA = {"binary_0.9": 0.90, "binary_0.75": 0.75}
print("profile:", RUN_PROFILE)


In [ ]:
# ============================================================
# 1) Portable Colab / Kaggle / local repository setup
# ============================================================
import json, os, shutil, subprocess, sys, time
from pathlib import Path

IN_COLAB  = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()

def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print(r.stderr[-1500:])
    if check and r.returncode != 0:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])

if IN_KAGGLE:   REPO_DIR = Path(REPO_DIR_KAGGLE); OUT = Path("/kaggle/working/c6_ceiling_gate")
elif IN_COLAB:  REPO_DIR = Path(REPO_DIR_COLAB);  OUT = Path("/content/c6_ceiling_gate")
else:           REPO_DIR = Path.cwd();            OUT = Path.cwd() / "c6_ceiling_gate"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))
OUT.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR, "| outputs:", OUT)


In [ ]:
# ============================================================
# 2) Load frozen c256 Pareto rows + import the ceiling kernel
# ============================================================
import numpy as np
from scripts.c6_perbatch_ceiling import (
    ceiling_worker, OBJECTIVE, TOTAL, MODE_TO_KAPPA, COUNT_TO_PAT,
)

pareto = json.loads((REPO_DIR / PARETO_REL).read_text())
rows = pareto["pareto_pairs"]
COMPARATOR = pareto["pareto"][0]["config_id"]
print("comparator:", COMPARATOR)
print("burned paired rows:", len(rows))
print("per-campaign per-batch calendars:", f"{TOTAL:,}")


In [ ]:
# ============================================================
# 3) Run the gate over all campaigns (parallel across campaigns)
# ============================================================
from concurrent.futures import ProcessPoolExecutor

sample = DEBUG_SAMPLE if RUN_PROFILE == "debug" else None
if RUN_PROFILE == "debug":
    # a couple from each stratum for a meaningful smoke test
    work = [r for r in rows if r["persistence_mode"] == "binary_0.9"][:DEBUG_CAMPAIGNS] + \
           [r for r in rows if r["persistence_mode"] == "binary_0.75"][:DEBUG_CAMPAIGNS]
else:
    work = rows
tasks = [{"history_root": int(r["history_root"]), "campaign_index": int(r["campaign_index"]),
          "persistence_mode": r["persistence_mode"], "frozen": float(r["retained"][OBJECTIVE]),
          "frozen_calendar": r["retained_calendar"], "sample": sample, "chunk": CHUNK} for r in work]
print(f"profile={RUN_PROFILE}  campaigns={len(tasks)}  "
      f"{'EXACT 8^8' if sample is None else f'sample={sample:,} (lower bound)'}  workers={N_WORKERS}")

t0 = time.perf_counter()
records = []
# ceiling_worker is a top-level function in the cloned repo -> picklable under fork and spawn.
try:
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        for i, out in enumerate(ex.map(ceiling_worker, tasks), 1):
            records.append(out)
            print(f"  [{i}/{len(tasks)}] root {out['history_root']} c{out['campaign_index']} "
                  f"{out['persistence_mode']}: perbatch {out['perbatch_ceiling']:.4f} "
                  f"coarse {out['coarse_ceiling']:.4f} frozen {out['frozen']:.4f} "
                  f"({time.perf_counter()-t0:.0f}s)")
except Exception as e:
    print("parallel pool failed, running serial:", e)
    records = [ceiling_worker(t) for t in tasks]
print(f"done in {time.perf_counter()-t0:.0f}s")


In [ ]:
# ============================================================
# 5) Integrity self-check  --  frozen replay must match the Pareto
# ============================================================
fails = [r for r in records if abs(r["replay"] - r["frozen"]) > 1e-9]
print(f"frozen-calendar replay self-check: {len(records)-len(fails)}/{len(records)} exact at 1e-9")
if fails:
    print("MISMATCHES (investigate before trusting the gate):")
    for r in fails[:5]:
        print("  ", r["history_root"], r["campaign_index"], r["replay"], r["frozen"])
assert not fails or RUN_PROFILE == "debug", "frozen replay mismatch on a serious run"


In [ ]:
# ============================================================
# 6) Clustered inference + verdict
# ============================================================
from collections import defaultdict

def clustered(deltas_by_root):
    roots = sorted(deltas_by_root)
    v = np.array([np.mean(deltas_by_root[r]) for r in roots], float)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    boot = np.array([rng.choice(v, size=len(v), replace=True).mean() for _ in range(BOOTSTRAP_DRAWS)])
    return {"mean": float(v.mean()), "lcb95_one_sided": float(np.quantile(boot, 0.05)),
            "ci95": [float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))], "n_roots": len(roots)}

strata = {}
for mode in ("binary_0.9", "binary_0.75", "POOLED"):
    subset = records if mode == "POOLED" else [r for r in records if r["persistence_mode"] == mode]
    cf, co = defaultdict(list), defaultdict(list)
    for r in subset:
        cf[r["history_root"]].append(r["perbatch_ceiling"] - r["frozen"])          # gate quantity
        co[r["history_root"]].append(r["perbatch_ceiling"] - r["coarse_ceiling"])   # ordering headroom
    strata[mode] = {"perbatch_ceiling_minus_frozen": clustered(cf),
                    "perbatch_ceiling_minus_coarse_ceiling": clustered(co),
                    "exact_optima_vs_perbatch": int(sum(
                        abs(r["perbatch_ceiling"] - r["frozen"]) <= 1e-12 for r in subset))}

opened = []
print("="*78)
print(f"C6 PER-BATCH CEILING GATE  --  profile={RUN_PROFILE}  ({'EXACT' if sample is None else 'SAMPLED lower bound'})")
print(f"gate: LCB95(perbatch_ceiling - frozen c256) >= {DEPLOYABLE_GATE}   SESOI={SESOI}")
print("="*78)
for mode, s in strata.items():
    g = s["perbatch_ceiling_minus_frozen"]; o = s["perbatch_ceiling_minus_coarse_ceiling"]
    door = g["lcb95_one_sided"] >= DEPLOYABLE_GATE
    if door and mode != "POOLED": opened.append(mode)
    print(f"\n[{mode}]  n_roots={g['n_roots']}  campaigns where frozen==perbatch ceiling: {s['exact_optima_vs_perbatch']}")
    print(f"   ceiling - frozen c256   mean {g['mean']:+.5f}  LCB95 {g['lcb95_one_sided']:+.5f}  "
          f"CI95 [{g['ci95'][0]:+.5f},{g['ci95'][1]:+.5f}]   gate {'PASS' if door else 'fail'}")
    print(f"   ceiling - coarse 4^8    mean {o['mean']:+.5f}  LCB95 {o['lcb95_one_sided']:+.5f}   "
          f"(value of intra-week ordering)")

print("\n" + "="*78)
if sample is not None:
    print("VERDICT (SAMPLED, lower bound only): a sample can OPEN the door but never CLOSE it.")
    print("  -> Re-run with RUN_PROFILE='serious' for the exact ceiling before concluding.")
elif opened:
    print(f"VERDICT: HEADROOM EXISTS in {opened} -> per-batch PPO is a justified experiment (Phase 2).")
    print("  The ceiling is clairvoyant: it proves a policy COULD exist, not that PPO finds it.")
else:
    print("VERDICT: DOOR CLOSED. No stratum clears the gate -> the exact best per-batch calendar")
    print("  cannot beat the frozen c256 comparator by a deployable margin. No learner can win in")
    print("  the per-batch space either. The last action-space door is closed WITHOUT training.")
print("="*78)


In [ ]:
# ============================================================
# 7) Save the receipt
# ============================================================
from datetime import datetime, timezone
receipt = {
    "schema_version": "q_r1_c6_perbatch_ceiling_gate_v1",
    "claim_status": "BURNED_DEVELOPMENT_NO_CLAIM",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "run_profile": RUN_PROFILE,
    "exact": sample is None,
    "sample_size": sample,
    "comparator": COMPARATOR,
    "objective": OBJECTIVE,
    "action_space": "per_batch_8^8 = 16,777,216",
    "gate_threshold": DEPLOYABLE_GATE,
    "sesoi": SESOI,
    "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED, "clustering": "history_root"},
    "strata": strata,
    "door_opened_strata": opened,
    "per_campaign": records,
    "git_branch": GIT_BRANCH,
}
path = OUT / f"c6_ceiling_gate_{RUN_PROFILE}.json"
path.write_text(json.dumps(receipt, indent=2, sort_keys=True, default=float))
print("saved:", path)
